# Random Forest Classifier

7-class classifier. Shallow/small forest (fast, small ONNX export, still accurate on flattened window features).

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================

%pip install pandas numpy scikit-learn joblib openpyxl matplotlib skl2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import export_text, plot_tree

In [ ]:
# ==========================================================
# CELL 2 — Random Forest configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s", "PWM", "Irms", "Vrms",
    "Current_Variance", "Current_Crest_Factor",
    "Vib_RMS", "Vib_Min", "Vib_Q1", "Vib_Median",
    "Vib_Q3", "Vib_Max", "Vib_Variance", "Vib_Skewness",
]

FINAL_LABELS = [
    "motor_off", "turning_off", "turning_on", "normal",
    "electrical_fault", "mechanical_fault", "thermal_fault",
]

LABEL_TO_ID = {label: index for index, label in enumerate(FINAL_LABELS)}

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)
WINDOW_LEN = 30
WINDOW_STEP = 10
INPUT_SIZE = WINDOW_LEN * N_FEATURES
ROW_PERIOD_SEC = 0.015
N_SPLITS = 5

OUTPUT_DIR = "random_forest_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Model: Random Forest")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Classes:", FINAL_LABELS)


In [ ]:
# ==========================================================
# CELL 3 — map each file path to one supervised class
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if any(x in path_text for x in [
        "motor_being_off", "turning_off", "transition_off", "off_spikes"
    ]):
        return "turning_off"

    if any(x in path_text for x in [
        "motor_being_on", "turning_on", "transition_on", "on_spikes"
    ]):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if any(x in path_text for x in [
        "electrical fault", "electrical_fault", "abnormal_current"
    ]):
        return "electrical_fault"

    if any(x in path_text for x in [
        "mechanical fault", "mechanical_fault", "abnormal_vibration"
    ]):
        return "mechanical_fault"

    if any(x in path_text for x in [
        "thermal fault", "thermal_fault", "abnormal_temp"
    ]):
        return "thermal_fault"

    if "/normal/" in path_text or stem.startswith("normal_") or stem == "normal":
        return "normal"

    return None


In [ ]:
# ==========================================================
# CELL 4 — discover and read CSV / Excel files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
    return sorted(set(files))


def read_one(path):
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower() == ".xlsx" else pd.read_csv(path)

    missing_columns = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in {path.name}: {missing_columns}")

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    clean = clean / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(f"{path.name} has only {len(clean)} valid rows.")

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)
if not files:
    raise RuntimeError(f"No CSV or Excel files found under: {BASE_PATH}")

sessions = []
for file_path in files:
    label = label_for(file_path)
    if label is None:
        print("SKIP unrecognized:", os.path.relpath(file_path, BASE_PATH))
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print("SKIP read error:", os.path.relpath(file_path, BASE_PATH), "|", error)
        continue

    sessions.append({
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": LABEL_TO_ID[label],
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError("No labeled sessions were loaded.")

class_counts = pd.Series([s["label"] for s in sessions]).value_counts().reindex(
    FINAL_LABELS, fill_value=0
)
print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [label for label in FINAL_LABELS if class_counts[label] == 0]
if missing_classes:
    raise RuntimeError(f"No files found for classes: {missing_classes}")


In [ ]:
# ==========================================================
# CELL 6 — create flattened 30 × 14 windows
# ==========================================================
def make_windows_from_array(X_raw):
    starts = np.arange(
        0,
        len(X_raw) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )
    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows.reshape(len(windows), INPUT_SIZE), starts


def build_dataset_from_sessions(selected_sessions):
    all_windows, all_labels, all_files, all_starts = [], [], [], []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(session["X"])
        if windows is None:
            continue

        all_windows.append(windows)
        all_labels.extend([session["class_id"]] * len(windows))
        all_files.extend([session["file"]] * len(windows))
        all_starts.extend(starts.tolist())

    if not all_windows:
        raise RuntimeError("No windows were created.")

    return (
        np.concatenate(all_windows, axis=0),
        np.asarray(all_labels, dtype=np.int64),
        np.asarray(all_files, dtype=object),
        np.asarray(all_starts, dtype=np.int64),
    )
    


In [ ]:
# ==========================================================
# CELL 7 — flattened feature names
# ==========================================================
FLAT_FEATURE_NAMES = [
    f"t{time_index:02d}_{feature_name}"
    for time_index in range(WINDOW_LEN)
    for feature_name in FEATURE_COLUMNS
]

In [ ]:
# ==========================================================
# CELL 8 — build Random Forest classifier
# ==========================================================
def build_classifier():
    return RandomForestClassifier(
        n_estimators=10,
        criterion="gini",
        max_depth=4,
        min_samples_split=20,
        min_samples_leaf=20,
        max_features="sqrt",
        bootstrap=True,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )


temporary_model = build_classifier()

print("Random Forest configuration:")
print(temporary_model.get_params())

del temporary_model


In [ ]:
# ==========================================================
# CELL 9 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [s["class_id"] for s in sessions],
    "group": [s["file"] for s in sessions],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42,
)

cv_metrics = []
all_true = []
all_pred = []
all_window_predictions = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(cv_table, cv_table["label"], groups=cv_table["group"]),
    start=1,
):
    print("\n" + "=" * 60)
    print("FOLD", fold)
    print("=" * 60)

    train_sessions = [sessions[index] for index in train_index]
    test_sessions = [sessions[index] for index in test_index]

    X_train, y_train, _, _ = build_dataset_from_sessions(train_sessions)
    X_test, y_test, test_files, test_starts = build_dataset_from_sessions(test_sessions)

    model = build_classifier()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    confidence = np.max(probabilities, axis=1)

    all_true.extend(y_test.tolist())
    all_pred.extend(predictions.tolist())

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "macro_precision": precision_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "mean_tree_depth": float(
            np.mean([
                estimator.tree_.max_depth
                for estimator in model.estimators_
            ])
        ),
        "maximum_tree_depth": int(
            np.max([
                estimator.tree_.max_depth
                for estimator in model.estimators_
            ])
        ),
        "mean_leaf_count": float(
            np.mean([
                estimator.tree_.n_leaves
                for estimator in model.estimators_
            ])
        ),
    }
    cv_metrics.append(fold_metrics)

    all_window_predictions.append(pd.DataFrame({
        "fold": fold,
        "file": test_files,
        "start_row": test_starts,
        "true_class_id": y_test,
        "true_label": [FINAL_LABELS[index] for index in y_test],
        "pred_class_id": predictions,
        "pred_label": [FINAL_LABELS[index] for index in predictions],
        "confidence": confidence,
    }))

    print(pd.Series(fold_metrics))

In [ ]:
# ==========================================================
# CELL 10 — combined cross-validation report
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_predictions_df = pd.concat(all_window_predictions, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_predictions_df.to_csv(
    os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False
)

all_true = np.asarray(all_true, dtype=np.int64)
all_pred = np.asarray(all_pred, dtype=np.int64)

overall_metrics = {
    "accuracy": accuracy_score(all_true, all_pred),
    "balanced_accuracy": balanced_accuracy_score(all_true, all_pred),
    "macro_precision": precision_score(
        all_true, all_pred, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        all_true, all_pred, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        all_true, all_pred, average="macro", zero_division=0
    ),
}

print("\nOVERALL CROSS-VALIDATION RESULTS")
print(pd.Series(overall_metrics))
print(classification_report(
    all_true,
    all_pred,
    labels=np.arange(N_CLASSES),
    target_names=FINAL_LABELS,
    zero_division=0,
))

pd.DataFrame([overall_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"), index=False
)

ConfusionMatrixDisplay(
    confusion_matrix(all_true, all_pred, labels=np.arange(N_CLASSES)),
    display_labels=FINAL_LABELS,
).plot(values_format="d", xticks_rotation=45)

plt.title("Random Forest Cross-Validation")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()


In [ ]:
# ==========================================================
# CELL 11 — train final embedded-friendly Random Forest
# ==========================================================
X_all, y_all, _, _ = (
    build_dataset_from_sessions(
        sessions
    )
)

final_model = build_classifier()

final_model.fit(
    X_all,
    y_all,
)

print(
    "Number of trees:",
    len(final_model.estimators_),
)

print(
    "Mean tree depth:",
    float(
        np.mean([
            estimator.tree_.max_depth
            for estimator in final_model.estimators_
        ])
    ),
)

print(
    "Maximum tree depth:",
    int(
        np.max([
            estimator.tree_.max_depth
            for estimator in final_model.estimators_
        ])
    ),
)

print(
    "Mean leaf count:",
    float(
        np.mean([
            estimator.tree_.n_leaves
            for estimator in final_model.estimators_
        ])
    ),
)

In [ ]:
# ==========================================================
# CELL 12 — save model, labels, metadata, and rules
# ==========================================================
MODEL_PATH = os.path.join(
    FINAL_DIR,
    "random_forest_classifier.pkl",
)

LABELS_PATH = os.path.join(
    FINAL_DIR,
    "random_forest_labels.txt",
)

METADATA_PATH = os.path.join(
    FINAL_DIR,
    "random_forest_metadata.csv",
)

RULES_PATH = os.path.join(
    FINAL_DIR,
    "random_forest_rules.txt",
)


# Save complete Random Forest model
joblib.dump(
    final_model,
    MODEL_PATH,
)


# Save class labels
with open(
    LABELS_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "\n".join(
            FINAL_LABELS
        )
    )


# Save flattened feature metadata
pd.DataFrame({
    "flat_feature_id": np.arange(
        INPUT_SIZE
    ),
    "flat_feature_name":
        FLAT_FEATURE_NAMES,
}).to_csv(
    METADATA_PATH,
    index=False,
)


# Save rules from every tree in the forest
with open(
    RULES_PATH,
    "w",
    encoding="utf-8",
) as file:

    for tree_index, estimator in enumerate(
        final_model.estimators_
    ):
        file.write(
            "\n"
            + "=" * 80
            + "\n"
        )

        file.write(
            f"RANDOM FOREST TREE {tree_index + 1}\n"
        )

        file.write(
            "=" * 80
            + "\n\n"
        )

        tree_rules = export_text(
            estimator,
            feature_names=
                FLAT_FEATURE_NAMES,
        )

        file.write(
            tree_rules
        )

        file.write(
            "\n\n"
        )


print("Saved:", MODEL_PATH)
print("Saved:", LABELS_PATH)
print("Saved:", METADATA_PATH)
print("Saved:", RULES_PATH)
print(
    "Exported trees:",
    len(final_model.estimators_),
)

In [ ]:
# ==========================================================
# CELL 13 — feature importance
# ==========================================================
importance_df = pd.DataFrame({
    "flat_feature_name": FLAT_FEATURE_NAMES,
    "importance": final_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

importance_df.to_csv(
    os.path.join(FINAL_DIR, "feature_importance.csv"), index=False
)
print("\nTOP 30 TREE FEATURES")
print(importance_df.head(30))


In [ ]:
# ==========================================================
# CELL 14 — first Random Forest tree visualization
# ==========================================================
plt.figure(figsize=(28, 14))
plot_tree(
    final_model.estimators_[0],
    feature_names=FLAT_FEATURE_NAMES,
    class_names=FINAL_LABELS,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=7,
)
plt.title("Random Forest — first tree, first 3 levels")
plt.tight_layout()
plt.savefig(
    os.path.join(FINAL_DIR, "random_forest_top_levels.png"), dpi=180
)
plt.show()


In [ ]:
# ==========================================================
# CELL 15 — export Cube.AI-compatible ONNX
# ==========================================================
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

ONNX_PATH = os.path.join(
    FINAL_DIR,
    "random_forest_classifier.onnx",
)

onnx_model = convert_sklearn(
    final_model,
    initial_types=[
        ("float_input", FloatTensorType([1, INPUT_SIZE]))
    ],
    target_opset=13,
    options={
        id(final_model): {
            "zipmap": False
        }
    },
)

with open(ONNX_PATH, "wb") as file:
    file.write(onnx_model.SerializeToString())

print("Saved:", ONNX_PATH)
print("Input shape: 1 x", INPUT_SIZE)
print("Class order:", final_model.classes_)

In [ ]:
# ==========================================================
# CELL 15A — generate and download Random Forest parameters
# ==========================================================
from pathlib import Path
import numpy as np

PARAM_HEADER_PATH = Path(FINAL_DIR) / (
    "random_forest_classifier_params.h"
)


def c_string(value):
    return str(value).replace("\\", "\\\\").replace('"', '\\"')


with open(
    PARAM_HEADER_PATH,
    "w",
    encoding="utf-8",
) as file:

    file.write(
r'''#ifndef RANDOM_FOREST_CLASSIFIER_PARAMS_H
#define RANDOM_FOREST_CLASSIFIER_PARAMS_H

#include <stdint.h>
#include <math.h>

/*
 * Parameters for:
 *     random_forest_classifier.onnx
 *
 * Model type:
 *     Seven-class Random Forest classifier
 *
 * Model input:
 *     1 x 420 FLOAT32
 *
 * Original window:
 *     30 time steps x 14 features
 *
 * IMPORTANT:
 * The original recorded dataset was divided by 1,000,000
 * when it was loaded in Python.
 *
 * Live STM32 features are already physical FLOAT32 values.
 * Therefore, do NOT divide the live STM32 values by
 * 1,000,000 before ONNX/X-CUBE-AI inference.
 *
 * No StandardScaler or MinMaxScaler is used.
 */

'''
    )

    file.write(
        f"#define RF_WINDOW_LEN       {WINDOW_LEN}U\n"
    )
    file.write(
        f"#define RF_N_FEATURES       {N_FEATURES}U\n"
    )
    file.write(
        f"#define RF_INPUT_SIZE       {INPUT_SIZE}U\n"
    )
    file.write(
        f"#define RF_N_CLASSES        {N_CLASSES}U\n"
    )
    file.write(
        f"#define RF_N_TREES          "
        f"{len(final_model.estimators_)}U\n"
    )
    file.write(
        f"#define RF_WINDOW_STEP      {WINDOW_STEP}U\n"
    )
    file.write(
        f"#define RF_ROW_PERIOD_SEC   {ROW_PERIOD_SEC:.10f}f\n"
    )
    file.write(
        "#define RF_WINDOW_DURATION_SEC "
        "(RF_WINDOW_LEN * RF_ROW_PERIOD_SEC)\n"
    )
    file.write(
        "#define RF_PREDICTION_INTERVAL_SEC "
        "(RF_WINDOW_STEP * RF_ROW_PERIOD_SEC)\n\n"
    )

    file.write(
r'''/*
 * Dataset-loading divisor retained only as metadata.
 * Do not apply this divisor to live STM32 physical values.
 */
#define RF_DATASET_SCALE_DIVISOR 1000000.0f
#define RF_APPLY_LIVE_DIVISOR    0U

'''
    )

    file.write(
        "/* Exact ONNX output-class order. */\n"
    )
    file.write(
        "static const char *const "
        "RF_LABELS[RF_N_CLASSES] = {\n"
    )

    for label in FINAL_LABELS:
        file.write(
            f'    "{c_string(label)}",\n'
        )

    file.write("};\n\n")

    file.write(
        "/* Exact feature order at every time step. */\n"
    )
    file.write(
        "static const char *const "
        "RF_FEATURE_NAMES[RF_N_FEATURES] = {\n"
    )

    for feature_name in FEATURE_COLUMNS:
        file.write(
            f'    "{c_string(feature_name)}",\n'
        )

    file.write("};\n\n")

    file.write(
r'''/*
 * Convert the 30 x 14 window to the flattened 420-value
 * input expected by the ONNX/X-CUBE-AI model.
 *
 * Flattening order:
 *     t0_feature0 ... t0_feature13,
 *     t1_feature0 ... t1_feature13,
 *     ...
 *     t29_feature0 ... t29_feature13
 */
static inline int rf_prepare_input(
    const float raw_window[RF_WINDOW_LEN][RF_N_FEATURES],
    float model_input[RF_INPUT_SIZE])
{
    if ((raw_window == 0) || (model_input == 0))
    {
        return -1;
    }

    for (uint32_t time_index = 0U;
         time_index < RF_WINDOW_LEN;
         time_index++)
    {
        for (uint32_t feature_index = 0U;
             feature_index < RF_N_FEATURES;
             feature_index++)
        {
            const uint32_t flat_index =
                time_index * RF_N_FEATURES
                + feature_index;

            const float value =
                raw_window[time_index][feature_index];

            if (!isfinite(value))
            {
                return -2;
            }

            /*
             * No scaling is performed here.
             * The STM32 value is already in physical units.
             */
            model_input[flat_index] = value;
        }
    }

    return 0;
}


/*
 * Find the predicted class and its confidence from the
 * seven probability outputs.
 */
static inline int rf_argmax(
    const float probability[RF_N_CLASSES],
    float *confidence)
{
    if ((probability == 0) || (confidence == 0))
    {
        return -1;
    }

    uint32_t predicted_class = 0U;
    float maximum_probability = probability[0];

    if (!isfinite(maximum_probability))
    {
        return -2;
    }

    for (uint32_t class_index = 1U;
         class_index < RF_N_CLASSES;
         class_index++)
    {
        if (!isfinite(probability[class_index]))
        {
            return -2;
        }

        if (probability[class_index] >
            maximum_probability)
        {
            maximum_probability =
                probability[class_index];

            predicted_class = class_index;
        }
    }

    *confidence = maximum_probability;

    return (int)predicted_class;
}

#endif /* RANDOM_FOREST_CLASSIFIER_PARAMS_H */
'''
    )

print("Created:", PARAM_HEADER_PATH)
print("Input shape: 1 x", INPUT_SIZE)
print("Number of classes:", N_CLASSES)
print("Number of trees:", len(final_model.estimators_))
print("Labels:", FINAL_LABELS)
print("No external MinMaxScaler or StandardScaler is required.")

# Automatically download in Google Colab
try:
    from google.colab import files
    files.download(str(PARAM_HEADER_PATH))
except ImportError:
    print(
        "Automatic download is available only in Google Colab."
    )
    print(
        "Saved header:",
        PARAM_HEADER_PATH.resolve()
    )

In [ ]:
# ==========================================================
# CELL 16 — export complete direct C header
# ==========================================================
HEADER_PATH = os.path.join(
    FINAL_DIR,
    "random_forest_classifier_model.h",
)

forest_trees = [
    estimator.tree_
    for estimator in final_model.estimators_
]

tree_offsets = []
node_offset = 0
all_feature = []
all_threshold = []
all_left = []
all_right = []
all_class = []

for tree in forest_trees:
    tree_offsets.append(node_offset)

    tree_class = np.argmax(
        tree.value[:, 0, :],
        axis=1,
    ).astype(np.int32)

    for node_index in range(tree.node_count):
        all_feature.append(int(tree.feature[node_index]))
        all_threshold.append(float(tree.threshold[node_index]))

        if tree.children_left[node_index] == -1:
            all_left.append(-1)
            all_right.append(-1)
        else:
            all_left.append(
                node_offset + int(tree.children_left[node_index])
            )
            all_right.append(
                node_offset + int(tree.children_right[node_index])
            )

        all_class.append(int(tree_class[node_index]))

    node_offset += tree.node_count

tree_offsets = np.asarray(tree_offsets, dtype=np.int32)
all_feature = np.asarray(all_feature, dtype=np.int16)
all_threshold = np.asarray(all_threshold, dtype=np.float32)
all_left = np.asarray(all_left, dtype=np.int32)
all_right = np.asarray(all_right, dtype=np.int32)
all_class = np.asarray(all_class, dtype=np.uint8)


def write_c_array(
    file,
    c_type,
    name,
    values,
    values_per_line=12,
):
    values = np.asarray(values).reshape(-1)

    file.write(
        f"static const {c_type} "
        f"{name}[{len(values)}] = {{\n"
    )

    for start in range(0, len(values), values_per_line):
        chunk = values[start:start + values_per_line]

        if "float" in c_type:
            line = ", ".join(
                f"{float(value):.9g}f"
                for value in chunk
            )
        else:
            line = ", ".join(
                str(int(value))
                for value in chunk
            )

        file.write(f"    {line},\n")

    file.write("};\n\n")


with open(
    HEADER_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write("#ifndef RANDOM_FOREST_CLASSIFIER_MODEL_H\n")
    file.write("#define RANDOM_FOREST_CLASSIFIER_MODEL_H\n\n")
    file.write("#include <stdint.h>\n\n")

    file.write(f"#define RF_WL {WINDOW_LEN}\n")
    file.write(f"#define RF_NF {N_FEATURES}\n")
    file.write(f"#define RF_INPUT {INPUT_SIZE}\n")
    file.write(f"#define RF_N_CLASSES {N_CLASSES}\n")
    file.write(f"#define RF_N_TREES {len(forest_trees)}\n")
    file.write(f"#define RF_TOTAL_NODES {len(all_feature)}\n")
    file.write(f"#define RF_DIV {SCALE_DIVISOR:.9g}f\n\n")

    write_c_array(file, "int32_t", "RF_TREE_OFFSET", tree_offsets)
    write_c_array(file, "int16_t", "RF_FEATURE", all_feature)
    write_c_array(file, "float", "RF_THRESHOLD", all_threshold)
    write_c_array(file, "int32_t", "RF_LEFT", all_left)
    write_c_array(file, "int32_t", "RF_RIGHT", all_right)
    write_c_array(file, "uint8_t", "RF_CLASS", all_class)

    c_source = r'''
static inline int random_forest_predict(
    const float raw[RF_WL][RF_NF],
    float probability[RF_N_CLASSES],
    float *confidence)
{
    uint16_t votes[RF_N_CLASSES] = {0};

    for (int tree_index = 0;
         tree_index < RF_N_TREES;
         tree_index++)
    {
        int32_t node = RF_TREE_OFFSET[tree_index];

        while (RF_LEFT[node] != -1)
        {
            int feature_id = RF_FEATURE[node];
            int time_index = feature_id / RF_NF;
            int feature_index = feature_id % RF_NF;

            float physical =
                raw[time_index][feature_index]
                / RF_DIV;

            if (physical <= RF_THRESHOLD[node])
            {
                node = RF_LEFT[node];
            }
            else
            {
                node = RF_RIGHT[node];
            }
        }

        votes[RF_CLASS[node]]++;
    }

    int class_id = 0;
    uint16_t maximum_votes = votes[0];

    for (int class_index = 0;
         class_index < RF_N_CLASSES;
         class_index++)
    {
        probability[class_index] =
            (float)votes[class_index]
            / (float)RF_N_TREES;

        if (votes[class_index] > maximum_votes)
        {
            maximum_votes = votes[class_index];
            class_id = class_index;
        }
    }

    *confidence = probability[class_id];

    return class_id;
}

#endif
'''

    file.write(c_source)

print("Saved:", HEADER_PATH)
print(
    "Header size KB:",
    round(os.path.getsize(HEADER_PATH) / 1024, 2),
)



In [ ]:
# ==========================================================
# CELL 17 — verify direct forest traversal
# ==========================================================
def direct_python_forest_predict(flattened_window):
    prob_sum = np.zeros(N_CLASSES, dtype=np.float64)

    for estimator in final_model.estimators_:
        tree = estimator.tree_
        node = 0
        while tree.children_left[node] != -1:
            feature_id = tree.feature[node]
            if flattened_window[feature_id] <= tree.threshold[node]:
                node = tree.children_left[node]
            else:
                node = tree.children_right[node]

        leaf_values = tree.value[node, 0, :]
        prob_sum += leaf_values / leaf_values.sum()

    return int(np.argmax(prob_sum))


number_check = min(100, len(X_all))

sklearn_predictions = final_model.predict(
    X_all[:number_check]
)

direct_predictions = np.asarray([
    direct_python_forest_predict(window)
    for window in X_all[:number_check]
])

if not np.array_equal(
    sklearn_predictions,
    direct_predictions,
):
    raise RuntimeError(
        "Direct forest traversal does not match "
        "the sklearn model."
    )

print(
    "Direct C-style forest traversal verified on",
    number_check,
    "windows.",
)



In [ ]:
# ==========================================================
# CELL 18 — manual classification test
# ==========================================================
def manual_classification_prediction(test_file):
    model = joblib.load(MODEL_PATH)
    test_df = read_one(test_file)
    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32)
    )

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    confidence = np.max(probabilities, axis=1)

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "pred_class_id": predictions,
        "pred_label": [FINAL_LABELS[index] for index in predictions],
        "confidence": confidence,
    })

    for class_id, label in enumerate(FINAL_LABELS):
        result[f"probability_{label}"] = probabilities[:, class_id]

    majority_class_id = int(result["pred_class_id"].value_counts().idxmax())
    majority_label = FINAL_LABELS[majority_class_id]
    stem = Path(test_file).stem

    output_csv = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}.csv",
    )
    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 60)
    print("MANUAL DECISION TREE TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Majority prediction:", majority_label)
    print("Mean confidence:", float(np.mean(confidence)))
    print("Saved CSV:", output_csv)
    print("\nWindow predictions:")
    print(result["pred_label"].value_counts())

    plt.figure(figsize=(12, 4))
    plt.step(
        result["time_sec"],
        result["pred_class_id"],
        where="mid",
    )
    plt.yticks(np.arange(N_CLASSES), FINAL_LABELS)
    plt.xlabel("Time [s]")
    plt.ylabel("Predicted class")
    plt.title(f"Random Forest — {stem}")
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_timeline.png",
        ),
        dpi=150,
    )
    plt.show()

    return result


In [ ]:
# ==========================================================
# CELL 19 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor CSV or Excel file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_classification_prediction(TEST_FILE)
    else:
        print("No file selected.")

